# Inference Pipeline — fs_hospital_weekly_demo
Dùng model đã train (preprocessor + XGBoost/LightGBM) để predict trên dữ liệu mới, lưu kết quả + SHAP vào `model_predictions_shap`.

In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery, storage
import pandas as pd, numpy as np, pickle, os, io, warnings
warnings.filterwarnings("ignore")

PROJECT_ID   = "project-8e2366a6-d3cc-40ee-9de"
DATASET_ID   = "hospital_feature_store"
SOURCE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.fs_hospital_weekly_demo"
OUTPUT_TABLE = f"{PROJECT_ID}.{DATASET_ID}.model_predictions_shap"

# GCS bucket chứa artifacts
BUCKET_NAME  = "project-8e2366a6-d3cc-40ee-9de-hospital-model"
GCS_PREFIX   = "hospital-model/xgboost"   # thư mục chứa pkl

bq = bigquery.Client(project=PROJECT_ID)
gcs = storage.Client(project=PROJECT_ID)
print(f"Source : {SOURCE_TABLE}")
print(f"Output : {OUTPUT_TABLE}")
print(f"Bucket : gs://{BUCKET_NAME}/{GCS_PREFIX}/")

Source : project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.fs_hospital_weekly_demo
Output : project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.model_predictions_shap_demo
Bucket : gs://project-8e2366a6-d3cc-40ee-9de-hospital-model/hospital-model/xgboost/


In [2]:
# ================================================================
# LOAD ARTIFACTS FROM GCS
# gs://BUCKET/hospital-model/xgboost/
#   preprocessor.pkl
#   model_reg.pkl   (XGBoost regressor)
#   model_cls.pkl   (XGBoost classifier)
#   explainer.pkl   (SHAP TreeExplainer) -- optional
# ================================================================
import shap

def load_pkl_from_gcs(bucket_name, blob_path):
    """Download a pickle file from GCS and return the deserialized object."""
    bucket = gcs.bucket(bucket_name)
    blob   = bucket.blob(blob_path)
    buf    = io.BytesIO()
    blob.download_to_file(buf)
    buf.seek(0)
    return pickle.load(buf)

def gcs_blob_exists(bucket_name, blob_path):
    return gcs.bucket(bucket_name).blob(blob_path).exists()

print("Loading artifacts from GCS...")

preprocessor = load_pkl_from_gcs(BUCKET_NAME, f"{GCS_PREFIX}/preprocessor.pkl")
print("  preprocessor.pkl  OK")

xgb_reg = load_pkl_from_gcs(BUCKET_NAME, f"{GCS_PREFIX}/model_reg.pkl")
print("  model_reg.pkl     OK")

xgb_cls = load_pkl_from_gcs(BUCKET_NAME, f"{GCS_PREFIX}/model_cls.pkl")
print("  model_cls.pkl     OK")

# SHAP explainer: load từ GCS nếu có, nếu không thì rebuild từ model
exp_xgb = None
if gcs_blob_exists(BUCKET_NAME, f"{GCS_PREFIX}/explainer.pkl"):
    exp_xgb = load_pkl_from_gcs(BUCKET_NAME, f"{GCS_PREFIX}/explainer.pkl")
    print("  explainer.pkl     OK (loaded from GCS)")
else:
    print("  explainer.pkl     not found -> rebuilding TreeExplainer...")
    exp_xgb = shap.TreeExplainer(xgb_reg)
    print("  TreeExplainer     rebuilt OK")

# LightGBM optional (trong hospital-model/lightgbm/)
lgb_reg, lgb_cls, exp_lgb = None, None, None
LGB_PREFIX = "hospital-model/lightgbm"
if gcs_blob_exists(BUCKET_NAME, f"{LGB_PREFIX}/model_reg.pkl"):
    lgb_reg = load_pkl_from_gcs(BUCKET_NAME, f"{LGB_PREFIX}/model_reg.pkl")
    lgb_cls = load_pkl_from_gcs(BUCKET_NAME, f"{LGB_PREFIX}/model_cls.pkl")
    if gcs_blob_exists(BUCKET_NAME, f"{LGB_PREFIX}/explainer.pkl"):
        exp_lgb = load_pkl_from_gcs(BUCKET_NAME, f"{LGB_PREFIX}/explainer.pkl")
    else:
        exp_lgb = shap.TreeExplainer(lgb_reg)
    print("  LightGBM artifacts loaded OK")
else:
    print("  LightGBM artifacts not found -> XGBoost only")

print("\nAll artifacts loaded successfully.")

Loading artifacts from GCS...
  preprocessor.pkl  OK
  model_reg.pkl     OK
  model_cls.pkl     OK
  explainer.pkl     OK (loaded from GCS)
  LightGBM artifacts loaded OK

All artifacts loaded successfully.


In [5]:
# ================================================================
# LOAD ARTIFACTS (preprocessor + models + SHAP explainers)
# ================================================================
def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

# preprocessor = load_pkl(f"{ARTIFACT_DIR}/preprocessor.pkl")
# xgb_reg      = load_pkl(f"{ARTIFACT_DIR}/xgb_reg_baseline.pkl")
# xgb_cls      = load_pkl(f"{ARTIFACT_DIR}/xgb_cls_baseline.pkl")

# # LightGBM optional — dùng nếu tồn tại
# # lgb_reg, lgb_cls = None, None
# # lgb_reg_path = f"{ARTIFACT_DIR}/lgb_reg_baseline.pkl"
# # lgb_cls_path = f"{ARTIFACT_DIR}/lgb_cls_baseline.pkl"
# # if os.path.exists(lgb_reg_path):
# #     lgb_reg = load_pkl(lgb_reg_path)
# #     lgb_cls = load_pkl(lgb_cls_path)
# #     print("Loaded: XGBoost + LightGBM")
# # else:
# #     print("Loaded: XGBoost only (LightGBM not found)")

# SHAP explainers — rebuild nếu pkl không có sẵn
import shap
print("Building SHAP TreeExplainer for XGBoost...")
exp_xgb = shap.TreeExplainer(xgb_reg)
if lgb_reg:
    print("Building SHAP TreeExplainer for LightGBM...")
    exp_lgb = shap.TreeExplainer(lgb_reg)

print("All artifacts loaded.")

Building SHAP TreeExplainer for XGBoost...
Building SHAP TreeExplainer for LightGBM...
All artifacts loaded.


In [13]:
# ================================================================
# LOAD NEW DATA FROM BIGQUERY
# ================================================================
print(f"Loading {SOURCE_TABLE}...")

raw_df = bq.query(f"SELECT * FROM `{SOURCE_TABLE}` ORDER BY hospital_id, report_date").to_dataframe()
raw_df["report_date"] = pd.to_datetime(raw_df["report_date"])

print(f"Rows     : {len(raw_df):,}")
print(f"Hospitals: {raw_df['hospital_id'].nunique():,}")
print(f"Date range: {raw_df['report_date'].min().date()} -> {raw_df['report_date'].max().date()}")
print(f"Columns  : {raw_df.shape[1]}")

# Quick sanity: check for -9999 sentinel
SENTINEL_COLS = ["icu_used","icu_occupancy_rate","covid_patients",
                 "covid_icu","flu_patients","flu_icu","covid_admit_adult"]
sentinel_ok = True
for col in SENTINEL_COLS:
    if col in raw_df.columns:
        n = (raw_df[col] == -9999).sum()
        if n > 0:
            print(f"  WARNING: {col} has {n} sentinel -9999 values -> converting to NaN")
            raw_df[col] = raw_df[col].replace(-9999, np.nan)
            sentinel_ok = False
if sentinel_ok:
    print("Sentinel check: OK (no -9999 values)")

Loading project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.fs_hospital_weekly_demo...
Rows     : 134,553
Hospitals: 1,300
Date range: 2023-11-05 -> 2026-04-18
Columns  : 71
Sentinel check: OK (no -9999 values)


In [7]:
# ================================================================
# ALIGN FEATURES với schema đã dùng lúc train
# ================================================================
EXCLUDE = [
    "hospital_id","report_date","county_fips","zip_code","county_name",
    "hospital_name","city","feature_computed_at",
    "target_occupancy_next_week","target_high_strain","occupancy_rate",
]
CATEGORICAL = ["state","hospital_type","season","disease_season",
               "healthcare_risk_level","metro_nonmetro_flag","hrr_region"]

# Lấy feature list từ preprocessor đã fit
try:
    num_feats = preprocessor.named_transformers_["num"].feature_names_in_.tolist()
    cat_feats = preprocessor.named_transformers_["cat"].feature_names_in_.tolist()
    FEATURE_COLS = num_feats + cat_feats
    print(f"Feature schema from preprocessor: {len(FEATURE_COLS)} features")
except Exception:
    # fallback: reconstruct từ columns
    FEATURE_COLS = [c for c in raw_df.columns if c not in EXCLUDE]
    num_feats    = [c for c in FEATURE_COLS if c not in CATEGORICAL]
    cat_feats    = [c for c in CATEGORICAL   if c in FEATURE_COLS]
    print(f"Feature schema reconstructed: {len(FEATURE_COLS)} features")

# Kiểm tra cột nào thiếu trong data mới
missing_cols = [c for c in FEATURE_COLS if c not in raw_df.columns]
if missing_cols:
    print(f"MISSING columns ({len(missing_cols)}): {missing_cols}")
    print("  -> Filling with NaN (preprocessor imputer will handle)")
    for c in missing_cols:
        raw_df[c] = np.nan
else:
    print("Column alignment: OK — all features present")

print(f"  Numeric : {len(num_feats)}")
print(f"  Categorical: {len(cat_feats)}")

ALL_FEAT_NAMES = num_feats + cat_feats

Feature schema from preprocessor: 61 features
Column alignment: OK — all features present
  Numeric : 54
  Categorical: 7


In [8]:
# ================================================================
# PREPROCESS + PREDICT
# ================================================================
print("Preprocessing new data...")
X_new    = preprocessor.transform(raw_df[FEATURE_COLS])
X_new_df = pd.DataFrame(X_new, columns=ALL_FEAT_NAMES)
print(f"X_new shape: {X_new.shape}")

# --- XGBoost predictions ---
pred_occ_xgb  = xgb_reg.predict(X_new).clip(0, 1)
pred_cls_xgb  = xgb_cls.predict(X_new)
pred_prob_xgb = xgb_cls.predict_proba(X_new)[:, 1]

print(f"XGBoost predictions done")
print(f"  Avg predicted occupancy : {pred_occ_xgb.mean():.1%}")
print(f"  High strain rate        : {pred_cls_xgb.mean():.1%}")

# --- LightGBM predictions (if available) ---
if lgb_reg:
    pred_occ_lgb  = lgb_reg.predict(X_new).clip(0, 1)
    pred_cls_lgb  = lgb_cls.predict(X_new)
    pred_prob_lgb = lgb_cls.predict_proba(X_new)[:, 1]
    print(f"LightGBM predictions done")
    print(f"  Avg predicted occupancy : {pred_occ_lgb.mean():.1%}")

Preprocessing new data...
X_new shape: (134553, 61)
XGBoost predictions done
  Avg predicted occupancy : 54.5%
  High strain rate        : 7.5%
LightGBM predictions done
  Avg predicted occupancy : 54.1%


In [9]:
# ================================================================
# COMPUTE SHAP VALUES
# ================================================================
print("Computing SHAP values for XGBoost...")
sv_xgb  = exp_xgb.shap_values(X_new_df)
base_xgb = float(exp_xgb.expected_value) if np.ndim(exp_xgb.expected_value)==0            else float(exp_xgb.expected_value[0])
print(f"  SHAP matrix: {sv_xgb.shape}")

if lgb_reg:
    print("Computing SHAP values for LightGBM...")
    sv_lgb  = exp_lgb.shap_values(X_new_df)
    base_lgb = float(exp_lgb.expected_value) if np.ndim(exp_lgb.expected_value)==0                else float(exp_lgb.expected_value[0])
    print(f"  SHAP matrix: {sv_lgb.shape}")

Computing SHAP values for XGBoost...
  SHAP matrix: (134553, 61)
Computing SHAP values for LightGBM...
  SHAP matrix: (134553, 61)


In [10]:
# ================================================================
# BUILD OUTPUT RECORDS (prediction + top-3 SHAP per row)
# ================================================================
from datetime import datetime

def make_records(model_name, pred_occ, pred_cls, pred_prob,
                 shap_vals, base_val, feat_names, df):
    rows = []
    for i in range(len(df)):
        sv   = shap_vals[i]
        top3 = np.argsort(np.abs(sv))[::-1][:3]
        arrow = lambda v: "up" if v > 0 else "down"

        # actual target nếu có trong data (có thể là NaN với dữ liệu demo)
        actual_occ    = float(df["target_occupancy_next_week"].iloc[i])                         if "target_occupancy_next_week" in df.columns else None
        actual_strain = int(df["target_high_strain"].iloc[i])                         if "target_high_strain" in df.columns else None
        abs_err = round(abs(actual_occ - pred_occ[i]), 4)                   if actual_occ is not None and not np.isnan(actual_occ) else None

        rows.append({
            "model_name"              : model_name,
            "hospital_id"             : str(df["hospital_id"].iloc[i]),
            "report_date"             : str(df["report_date"].iloc[i])[:10],
            "pred_occupancy_next_week": round(float(pred_occ[i]), 4),
            "pred_high_strain"        : int(pred_cls[i]),
            "pred_high_strain_prob"   : round(float(pred_prob[i]), 4),
            "actual_occupancy"        : round(actual_occ, 4) if actual_occ and not np.isnan(actual_occ) else None,
            "actual_high_strain"      : actual_strain,
            "abs_error"               : abs_err,
            "shap_base_value"         : round(base_val, 4),
            "top1_feature"            : feat_names[top3[0]],
            "top1_shap"               : round(float(sv[top3[0]]), 4),
            "top1_direction"          : arrow(sv[top3[0]]),
            "top2_feature"            : feat_names[top3[1]],
            "top2_shap"               : round(float(sv[top3[1]]), 4),
            "top2_direction"          : arrow(sv[top3[1]]),
            "top3_feature"            : feat_names[top3[2]],
            "top3_shap"               : round(float(sv[top3[2]]), 4),
            "top3_direction"          : arrow(sv[top3[2]]),
            "explanation"             : (
                f"{feat_names[top3[0]]} ({arrow(sv[top3[0]])} {abs(sv[top3[0]]):.3f}), "
                f"{feat_names[top3[1]]} ({arrow(sv[top3[1]])} {abs(sv[top3[1]]):.3f}), "
                f"{feat_names[top3[2]]} ({arrow(sv[top3[2]])} {abs(sv[top3[2]]):.3f})"
            ),
            "data_source"             : "fs_hospital_weekly_demo",
            "predicted_at"            : datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        })
    return pd.DataFrame(rows)

# XGBoost records
records_xgb = make_records(
    "XGBoost", pred_occ_xgb, pred_cls_xgb, pred_prob_xgb,
    sv_xgb, base_xgb, ALL_FEAT_NAMES, raw_df
)

# LightGBM records
all_records = records_xgb.copy()
if lgb_reg:
    records_lgb = make_records(
        "LightGBM", pred_occ_lgb, pred_cls_lgb, pred_prob_lgb,
        sv_lgb, base_lgb, ALL_FEAT_NAMES, raw_df
    )
    all_records = pd.concat([records_xgb, records_lgb], ignore_index=True)

print(f"Total prediction records: {len(all_records):,}")
print(f"Models: {all_records['model_name'].unique().tolist()}")
print(f"\nSample high-strain alert (XGBoost):")
hs = all_records[(all_records.model_name=="XGBoost") & (all_records.pred_high_strain==1)].head(2)
if len(hs):
    for _, r in hs.iterrows():
        print(f"  {r.hospital_id} | pred={r.pred_occupancy_next_week:.1%} | prob={r.pred_high_strain_prob:.2f}")
        print(f"  Explanation: {r.explanation}")
else:
    print("  (no high-strain predictions in sample)")

Total prediction records: 269,106
Models: ['XGBoost', 'LightGBM']

Sample high-strain alert (XGBoost):
  010039 | pred=87.5% | prob=0.63
  Explanation: occ_roll4 (up 0.159), occ_roll8 (up 0.038), occ_lag3 (down 0.013)
  010039 | pred=88.7% | prob=0.83
  Explanation: occ_roll4 (up 0.143), occ_lag1 (up 0.038), occ_roll8 (up 0.035)


In [15]:
# ================================================================
# UPLOAD TO BIGQUERY: model_predictions_shap
# Write mode: WRITE_APPEND — giữ lại kết quả cũ (train set)
#             dùng WRITE_TRUNCATE nếu muốn ghi đè toàn bộ
# ================================================================
OUTPUT_TABLE = f"{PROJECT_ID}.{DATASET_ID}.model_predictions_shap_demo"
job_cfg = bigquery.LoadJobConfig(
    write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect        = True,
)

job = bq.load_table_from_dataframe(all_records, OUTPUT_TABLE, job_config=job_cfg)
job.result()

tbl = bq.get_table(OUTPUT_TABLE)
print(f"Uploaded to: {OUTPUT_TABLE}")
print(f"Total rows in table: {tbl.num_rows:,}")


Uploaded to: project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.model_predictions_shap_demo
Total rows in table: 269,106


In [18]:
# ================================================================
# QUICK VISUAL: distribution of predictions on demo data
# ================================================================
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

xgb_preds = all_records[all_records.model_name == "XGBoost"].copy()
xgb_preds["report_date"] = pd.to_datetime(xgb_preds["report_date"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Inference results on fs_hospital_weekly_demo (XGBoost)", fontsize=11)

# Histogram of predicted occupancy
ax = axes[0]
ax.hist(xgb_preds["pred_occupancy_next_week"]*100, bins=30,
        color="#2E75B6", edgecolor="white", linewidth=0.4)
ax.axvline(85, color="#E24B4A", linestyle="--", linewidth=1, label="85% threshold")
ax.set_title("Predicted occupancy distribution")
ax.set_xlabel("Predicted occupancy (%)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Top features by mean |SHAP|
ax = axes[1]
shap_means = {}
for feat in ALL_FEAT_NAMES:
    # compute mean |SHAP| from records
    pass
top_feat_counts = (xgb_preds["top1_feature"].value_counts() +
                   xgb_preds["top2_feature"].value_counts().fillna(0) +
                   xgb_preds["top3_feature"].value_counts().fillna(0)).fillna(0)
top10 = top_feat_counts.sort_values(ascending=False).head(10)
top10.sort_values().plot.barh(ax=ax, color="#534AB7", alpha=0.85)
ax.set_title("Top features (frequency in top-3 SHAP)")
ax.set_xlabel("Count"); ax.grid(True, alpha=0.3)

# Weekly trend of high-strain rate
ax = axes[2]
weekly = xgb_preds.groupby(xgb_preds["report_date"].dt.to_period("W")).agg(
    hs_rate=("pred_high_strain","mean")).reset_index()
weekly["report_date"] = weekly["report_date"].dt.to_timestamp()
ax.plot(weekly["report_date"], weekly["hs_rate"]*100, color="#E24B4A", lw=1.5)
ax.set_title("High-strain rate over time (%)"); ax.set_ylabel("%")
ax.tick_params(axis="x", rotation=30); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/inference_results.png", dpi=130, bbox_inches="tight")
plt.show()

print(f"\nSummary:")
print(f"  Predictions   : {len(xgb_preds):,}")
print(f"  Avg occupancy : {xgb_preds.pred_occupancy_next_week.mean():.1%}")
print(f"  High strain   : {xgb_preds.pred_high_strain.mean():.1%}")
print(f"  Chart saved   : /content/inference_results.png")
print(f"\nDone! Query Looker Studio from: {OUTPUT_TABLE}")


Summary:
  Predictions   : 134,553
  Avg occupancy : 54.5%
  High strain   : 7.5%
  Chart saved   : /content/inference_results.png

Done! Query Looker Studio from: project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.model_predictions_shap_demo
